# MCP Resources and Prompts

**Phase 03** · ~60 minutes · Python

**Prerequisites:** See lesson README

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thepandanlabs/applied-ai-from-scratch/blob/main/notebooks/phase-03/10-mcp-resources-and-prompts.ipynb)

---

> This notebook is auto-generated from the [Applied AI From Scratch](https://github.com/thepandanlabs/applied-ai-from-scratch) curriculum.  
> Run cells top to bottom. Set your API key in the **Setup** cell below.


In [ ]:
!pip install -q mcp

import os

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except Exception:
    pass  # Running locally — set OPENAI_API_KEY before this cell

print("Setup complete. API key set:", bool(os.environ.get("OPENAI_API_KEY")))

## Implementation

Run each cell in order:

### Setup & Imports

In [ ]:
"""
L10: MCP Resources and Prompts

Demonstrates the three MCP primitives: tools, resources, and prompts.
Extends the product database server with:
  - A static schema resource (docs://products/schema)
  - A dynamic product resource (product://{product_id})
  - A catalog resource (docs://products/catalog)
  - A sales analysis prompt template (analyze_sales)

Run this server with stdio transport and connect via MCP Inspector or
the test client at the bottom of this file.

Usage:
    # Start the server (for MCP Inspector or Claude Desktop):
    python main.py --server

    # Run the demo client (shows resource reads and prompt rendering):
    python main.py --client

    # Run smoke tests:
    python main.py --test
"""

import argparse
import asyncio
import json
import sys
from mcp.server import FastMCP

### Server setup

In [ ]:
mcp = FastMCP("product-database")

PRODUCTS = {
    "p001": {"name": "Widget A", "price": 9.99, "stock": 142, "category": "hardware"},
    "p002": {"name": "Widget B", "price": 24.99, "stock": 8, "category": "hardware"},
    "p003": {"name": "Gadget X", "price": 149.00, "stock": 0, "category": "electronics"},
    "p004": {"name": "Gadget Y", "price": 89.00, "stock": 37, "category": "electronics"},
}

SALES = [
    {"product_id": "p001", "date": "2025-05-01", "units": 12, "revenue": 119.88},
    {"product_id": "p002", "date": "2025-05-01", "units": 3, "revenue": 74.97},
    {"product_id": "p001", "date": "2025-05-15", "units": 8, "revenue": 79.92},
    {"product_id": "p003", "date": "2025-05-20", "units": 1, "revenue": 149.00},
]

### Tools

In [ ]:
@mcp.tool()
def search_products(query: str) -> list[dict]:
    """Search products by name or category."""
    q = query.lower()
    return [
        {"id": k, **v}
        for k, v in PRODUCTS.items()
        if q in v["name"].lower() or q in v["category"].lower()
    ]


@mcp.tool()
def get_sales_summary() -> dict:
    """Get aggregated sales totals across all products."""
    total_revenue = sum(s["revenue"] for s in SALES)
    total_units = sum(s["units"] for s in SALES)
    by_product: dict[str, dict] = {}
    for sale in SALES:
        pid = sale["product_id"]
        if pid not in by_product:
            by_product[pid] = {"units": 0, "revenue": 0.0}
        by_product[pid]["units"] += sale["units"]
        by_product[pid]["revenue"] += sale["revenue"]
    return {
        "total_revenue": round(total_revenue, 2),
        "total_units": total_units,
        "by_product": by_product,
    }

### Resources

In [ ]:
DB_SCHEMA = """
Products table:
  product_id  TEXT PRIMARY KEY
  name        TEXT NOT NULL
  price       REAL NOT NULL
  stock       INTEGER NOT NULL
  category    TEXT NOT NULL

Sales table:
  id          INTEGER PRIMARY KEY
  product_id  TEXT REFERENCES products(product_id)
  date        TEXT  -- ISO 8601: YYYY-MM-DD
  units       INTEGER
  revenue     REAL
"""


@mcp.resource("docs://products/schema")
def get_schema() -> str:
    """The database schema for the product and sales tables."""
    return DB_SCHEMA


@mcp.resource("docs://products/catalog")
def get_catalog() -> str:
    """All products as JSON, suitable for injection into LLM context."""
    return json.dumps(
        [{"id": k, **v} for k, v in PRODUCTS.items()],
        indent=2,
    )


@mcp.resource("product://{product_id}")
def get_product_resource(product_id: str) -> str:
    """Product details as a formatted text block, addressed by URI template."""
    if product_id not in PRODUCTS:
        return f"Product {product_id} not found."
    p = PRODUCTS[product_id]
    return (
        f"Product: {p['name']}\n"
        f"ID: {product_id}\n"
        f"Price: ${p['price']:.2f}\n"
        f"Stock: {p['stock']} units\n"
        f"Category: {p['category']}\n"
        f"Status: {'In stock' if p['stock'] > 0 else 'Out of stock'}"
    )

### Prompts

In [ ]:
@mcp.prompt()
def analyze_sales(time_period: str) -> str:
    """
    Generate a sales analysis prompt for a given time period.
    Returns a fully-formed prompt ready to send to an LLM.
    """
    sales_data = json.dumps(SALES, indent=2)
    return (
        f"Analyze the following sales data for {time_period}.\n\n"
        f"Sales records:\n{sales_data}\n\n"
        "Your analysis should cover:\n"
        "1. Total revenue and unit volume\n"
        "2. Best-performing product by revenue\n"
        "3. Any products with concerning stock levels (below 10 units)\n"
        "4. One actionable recommendation for the next period\n\n"
        "Be concise. Use bullet points for the findings."
    )

### Demo client

In [ ]:
async def run_demo_client():
    """Connect to the server and demonstrate reading resources and prompts."""
    from mcp import ClientSession, StdioServerParameters
    from mcp.client.stdio import stdio_client

    server_params = StdioServerParameters(
        command=sys.executable,
        args=[__file__, "--server"],
    )

    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            # List resources
            resources = await session.list_resources()
            print("=== Available Resources ===")
            for r in resources.resources:
                print(f"  {r.uri}")
                if r.description:
                    print(f"    {r.description}")

            # Read static resource
            print("\n=== Schema Resource (docs://products/schema) ===")
            schema = await session.read_resource("docs://products/schema")
            print(schema.contents[0].text.strip())

            # Read dynamic resource via URI template
            print("\n=== Product Resource (product://p001) ===")
            product = await session.read_resource("product://p001")
            print(product.contents[0].text)

            # Read unknown product
            print("\n=== Product Resource (product://unknown) ===")
            unknown = await session.read_resource("product://unknown")
            print(unknown.contents[0].text)

            # List prompts
            prompts = await session.list_prompts()
            print("\n=== Available Prompts ===")
            for p in prompts.prompts:
                args = [f"{a.name} (required={a.required})" for a in (p.arguments or [])]
                print(f"  {p.name}: args = {args}")

            # Render the prompt
            print("\n=== Rendered Prompt: analyze_sales(time_period='May 2025') ===")
            rendered = await session.get_prompt(
                "analyze_sales",
                {"time_period": "May 2025"},
            )
            print(rendered.messages[0].content.text[:400])
            print("...(truncated)")


async def run_smoke_tests():
    """Verify all resources and prompts are registered and return expected content."""
    from mcp import ClientSession, StdioServerParameters
    from mcp.client.stdio import stdio_client

    server_params = StdioServerParameters(
        command=sys.executable,
        args=[__file__, "--server"],
    )

    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            # Resources present
            resources = await session.list_resources()
            uris = [r.uri for r in resources.resources]
            assert "docs://products/schema" in uris, f"Schema resource missing. Got: {uris}"
            assert "docs://products/catalog" in uris, f"Catalog resource missing. Got: {uris}"

            # Schema content
            schema = await session.read_resource("docs://products/schema")
            assert "Products table" in schema.contents[0].text, "Schema content wrong"

            # Dynamic resource: valid product
            p1 = await session.read_resource("product://p001")
            assert "Widget A" in p1.contents[0].text, "p001 content wrong"

            # Dynamic resource: invalid product
            pnone = await session.read_resource("product://does-not-exist")
            assert "not found" in pnone.contents[0].text.lower(), "Invalid product error message wrong"

            # Prompt registered
            prompts = await session.list_prompts()
            names = [p.name for p in prompts.prompts]
            assert "analyze_sales" in names, f"analyze_sales prompt missing. Got: {names}"

            # Prompt renders with argument substitution
            rendered = await session.get_prompt("analyze_sales", {"time_period": "Q2 2025"})
            assert "Q2 2025" in rendered.messages[0].content.text, "Prompt argument not substituted"

    print("All smoke tests passed.")


# ---------------------------------------------------------------------------
# Entry point
# ---------------------------------------------------------------------------

### Demo

In [ ]:
parser = argparse.ArgumentParser(description="MCP resources and prompts demo")
parser.add_argument("--server", action="store_true", help="Run as MCP server (stdio)")
parser.add_argument("--client", action="store_true", help="Run demo client")
parser.add_argument("--test", action="store_true", help="Run smoke tests")
args = parser.parse_args()

if args.server:
    mcp.run(transport="stdio")
elif args.client:
    asyncio.run(run_demo_client())
elif args.test:
    asyncio.run(run_smoke_tests())
else:
    # Default: run demo client
    asyncio.run(run_demo_client())

---

## Self-Check

Answer these without running code first:

**1. What is the most significant cost of implementing this as a tool instead of a resource?**

- A. Tools are slower than resources because they run in a separate thread.
- B. Each schema retrieval requires an LLM API call to decide to invoke the tool, adding latency and cost for data the client could read directly.
- C. Tools cannot return string data; they must return structured JSON.
- D. The schema content will be truncated by the tool output size limit.

**2. What should the resource handler return, and why?**

- A. Raise a KeyError exception so the client knows the ID is invalid.
- B. Return an empty string so the client receives no content for missing products.
- C. Return a descriptive 'not found' message string so the LLM can reason about the absence.
- D. Return None to signal the resource does not exist.

**3. What goes wrong with this approach in production?**

- A. Module-level fetches fail because the database is not connected at import time.
- B. The ticket count is captured once at startup and never updated, so every rendered prompt shows the count from when the server last restarted.
- C. Prompt arguments cannot be used alongside module-level data in the same template.
- D. The template string becomes too large for the MCP protocol to transmit.

**4. How does defining the prompt on the MCP server instead solve the divergence problem?**

- A. MCP prompts enforce a schema that prevents teams from modifying the prompt.
- B. The prompt template lives on the server. All clients render the same template. Updating the server updates all clients simultaneously with no client redeploy.
- C. MCP prompts are version-controlled by the protocol, so teams always see the latest version.
- D. Server-side prompts cannot be copy-pasted, which forces teams to use the canonical version.

**5. Which is more appropriate and why?**

- A. Resource, because resources are faster than tools.
- B. Tool, because the LLM can pass parameters to filter the data and get only what it needs.
- C. Resource, because resources are designed for frequently changing data.
- D. Either works equally well for frequently changing data.

**6. What is the most likely cause?**

- A. URI templates are not included in list_resources() because the full list of possible URIs is infinite.
- B. The product resource handler raised an exception during registration.
- C. URI templates must be registered separately using list_resource_templates() not list_resources().
- D. The product URI scheme is not recognized by the MCP protocol.

**7. What is the root cause and the right fix?**

- A. The resource data is wrong. Update the jurisdiction guides.
- B. Claude Desktop cached the resource content at session start. Stale resources require either manual session restart or a resource subscription mechanism to push updates to the client.
- C. Resources should not be used for legal data; use tools instead.
- D. The LLM is ignoring the resource content and generating from training data.

_Answers are in `checks.json` in the lesson directory._